# <font color="#418FDE" size="6.5" uppercase>**Workflow and Baselines**</font>

>Last update: 20260708.
    
By the end of this Lecture, you will be able to:
- Describe the steps in an end-to-end machine learning workflow for civil engineering data. 
- Implement a manual train/test split using simple Python and NumPy operations. 
- Build baseline predictors for regression and classification before training complex models. 


## **1. ML Pipeline**

### **1.1. Problem Framing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_01_01.jpg?v=1783555158" width="250">



>* Turn engineering needs into precise prediction tasks
>* Define outcomes, scope, timing, and use

>* Match task type to data and models
>* Define success using engineering risk context

>* Account for data limits and assumptions
>* Define users, inputs, actions, and errors



### **1.2. Data Preparation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_01_02.jpg?v=1783555160" width="250">



>* Organize messy engineering data from many sources
>* Align records and variables for reliable modeling

>* Find and judge data quality problems
>* Document fixes for transparent, safe modeling

>* Create useful features from engineering data
>* Prevent leakage for fair model evaluation



In [ ]:
#@title Python Code - Data Preparation

# Prepare civil engineering data before modeling.
# Clean units, missing values, and duplicates.
# Create simple features for fair evaluation.

# Import small, standard teaching libraries.
import numpy as np
import pandas as pd

# Build a tiny bridge inspection dataset.
data = {
    "bridge_id": [101, 102, 103, 103, 104, 105],
    "span_ft": [120, 85, 200, 200, 60, 150],
    "crack_cm": [0.4, np.nan, 1.8, 1.8, 0.2, 9.9],
    "age_years": [12, 35, 48, 48, 8, 60],
    "condition": ["good", "fair", "poor", "poor", "good", "poor"],
}

# Convert the dictionary into tabular data.
df = pd.DataFrame(data)

# Remove repeated inspection records safely.
df = df.drop_duplicates(subset=["bridge_id"])

# Convert span from feet to meters.
df["span_m"] = df["span_ft"] * 0.3048

# Flag unusually large crack measurements.
df["crack_flag"] = df["crack_cm"] > 5.0

# Impute missing crack width using median.
median_crack = df["crack_cm"].median()
df["crack_cm"] = df["crack_cm"].fillna(median_crack)

# Create an engineering feature for modeling.
df["age_per_span"] = df["age_years"] / df["span_m"]

# Keep only clear modeling columns.
prepared = df[["span_m", "crack_cm", "age_years", "age_per_span", "condition"]]

# Validate the prepared dataset shape.
assert prepared.shape == (5, 5)

# Print a compact preparation summary.
print("Rows after duplicate removal:", len(prepared))
print("Median crack used for imputation:", round(median_crack, 2))
print("Flagged possible sensor or entry errors:", int(df["crack_flag"].sum()))
print("Prepared columns:", list(prepared.columns))
print("First prepared record:", prepared.iloc[0].to_dict())



### **1.3. Train Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_01_03.jpg?v=1783555162" width="250">



>* Models learn patterns from prepared engineering data
>* Training supports reliable decisions on unseen cases

>* Start with baselines before complex models
>* Validate models on unseen engineering cases

>* Evaluate errors across uncertain site conditions
>* Iterate models for responsible engineering decisions



In [ ]:
#@title Python Code - Train Models

# Train simple models for engineering prediction.
# Compare baselines before adding model complexity.
# Use NumPy only for transparent learning.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for repeatable results.
rng = np.random.default_rng(42)

# Create tiny pavement deterioration training data.
age_years = np.array([1, 2, 3, 4, 5, 6, 7, 8])
traffic_kips = np.array([12, 15, 18, 22, 25, 29, 33, 36])

# Define target condition loss values.
condition_loss = np.array([4, 7, 10, 15, 18, 23, 27, 31])

# Validate matching feature and target lengths.
assert age_years.size == condition_loss.size
assert traffic_kips.size == condition_loss.size

# Build a simple design matrix.
ones_column = np.ones_like(age_years)
X = np.column_stack([ones_column, age_years, traffic_kips])

# Manually split rows into train and test.
indices = rng.permutation(X.shape[0])
train_idx = indices[:6]
test_idx = indices[6:]

# Separate training and testing arrays.
X_train = X[train_idx]
y_train = condition_loss[train_idx]
X_test = X[test_idx]
y_test = condition_loss[test_idx]

# Train a baseline using training mean.
baseline_value = y_train.mean()
baseline_pred = np.full(y_test.shape, baseline_value)

# Train linear regression using least squares.
weights = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
model_pred = X_test @ weights

# Define mean absolute error calculation.
def mean_absolute_error(actual, predicted):
    return np.mean(np.abs(actual - predicted))

# Evaluate baseline and trained model.
baseline_mae = mean_absolute_error(y_test, baseline_pred)
model_mae = mean_absolute_error(y_test, model_pred)

# Print compact workflow results.
print(f"NumPy version: {np.__version__}")
print(f"Training rows: {X_train.shape[0]}, test rows: {X_test.shape[0]}")
print(f"Baseline MAE: {baseline_mae:.2f} condition-loss points")
print(f"Linear model MAE: {model_mae:.2f} condition-loss points")
print("Baseline first, then train candidate models.")

# Plot actual and predicted test values.
plt.figure(figsize=(6, 4))
plt.scatter(y_test, baseline_pred, label="Baseline", s=80)
plt.scatter(y_test, model_pred, label="Linear model", s=80)
plt.plot([0, 35], [0, 35], "k--", label="Perfect prediction")
plt.xlabel("Actual condition loss")
plt.ylabel("Predicted condition loss")
plt.title("Training Models After a Manual Split")
plt.legend()
plt.tight_layout()
plt.show()



## **2. Manual Data Splitting**

### **2.1. Randomized Splitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_02_01.jpg?v=1783555164" width="250">



>* Shuffle records before splitting datasets
>* Create representative train and test sets

>* Shuffle row positions before selecting subsets
>* Create balanced training and independent test sets

>* Use fixed seeds for repeatable splits
>* Avoid random splits for dependent data



In [ ]:
#@title Python Code - Randomized Splitting

# Demonstrate randomized splitting with NumPy.
# Use small civil engineering sample data.
# Keep results reproducible with seeds.

# Import NumPy for arrays and randomization.
import numpy as np

# Create a reproducible random number generator.
rng = np.random.default_rng(seed=42)

# Store bridge identifiers as simple labels.
bridge_ids = np.array(["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08"])

# Store bridge ages in years.
bridge_age_years = np.array([12, 35, 18, 50, 7, 28, 41, 22])

# Store inspection ratings from one to five.
condition_rating = np.array([4, 2, 5, 1, 5, 3, 2, 4])

# Validate that all arrays share length.
row_count = len(bridge_ids)

# Stop early if arrays are inconsistent.
assert len(bridge_age_years) == row_count
assert len(condition_rating) == row_count

# Choose the test fraction for splitting.
test_fraction = 0.25

# Convert the fraction into row counts.
test_count = int(round(row_count * test_fraction))

# Keep at least one training row.
test_count = min(max(test_count, 1), row_count - 1)

# Create shuffled row positions.
shuffled_indices = rng.permutation(row_count)

# Select test rows from shuffled positions.
test_indices = shuffled_indices[:test_count]

# Select training rows from remaining positions.
train_indices = shuffled_indices[test_count:]

# Use indices to gather training data.
train_ids = bridge_ids[train_indices]
train_ages = bridge_age_years[train_indices]
train_ratings = condition_rating[train_indices]

# Use indices to gather testing data.
test_ids = bridge_ids[test_indices]
test_ages = bridge_age_years[test_indices]
test_ratings = condition_rating[test_indices]

# Print a compact summary for learners.
print("Manual randomized train/test split")
print(f"Total rows: {row_count}")
print(f"Training rows: {len(train_indices)}")
print(f"Testing rows: {len(test_indices)}")

# Show the shuffled row order.
print(f"Shuffled indices: {shuffled_indices.tolist()}")

# Show selected bridge identifiers only.
print(f"Train bridge IDs: {train_ids.tolist()}")
print(f"Test bridge IDs: {test_ids.tolist()}")

# Compare simple target averages.
print(f"Train average rating: {train_ratings.mean():.2f}")
print(f"Test average rating: {test_ratings.mean():.2f}")

# Confirm no row appears in both groups.
overlap_count = len(set(train_indices).intersection(set(test_indices)))
print(f"Overlapping rows: {overlap_count}")



### **2.2. Test Set Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_02_02.jpg?v=1783555168" width="250">



>* Hold out data for final evaluation
>* Represent future unseen civil engineering cases

>* Choose a test size that preserves training data
>* Keep test data balanced and representative

>* Prevent test data from shaping training choices
>* Split by real prediction units when needed



In [ ]:
#@title Python Code - Test Set Selection

# Manual splitting supports honest model evaluation.
# Test rows must stay unseen.
# Random seeds make examples repeatable.

import numpy as np
import pandas as pd

# Create small civil engineering inspection data.
bridge_ids = np.array([101, 102, 103, 104, 105, 106, 107, 108])
condition = np.array([72, 55, 88, 61, 47, 93, 69, 58])
traffic = np.array([12, 35, 8, 22, 41, 6, 18, 29])

# Store rows in a compact table.
data = pd.DataFrame({"bridge_id": bridge_ids, "condition": condition, "traffic_k": traffic})

# Choose a reproducible random generator.
rng = np.random.default_rng(seed=42)

# Shuffle row positions, not the data values.
row_count = len(data)
indices = np.arange(row_count)
shuffled = rng.permutation(indices)

# Reserve one quarter of rows for testing.
test_fraction = 0.25
test_count = max(1, int(round(row_count * test_fraction)))

# Validate the split size before slicing.
assert 0 < test_count < row_count, "Test size must be valid."

# Select test rows first, then training rows.
test_indices = shuffled[:test_count]
train_indices = shuffled[test_count:]

# Build separate tables using selected positions.
test_set = data.iloc[test_indices].sort_index()
train_set = data.iloc[train_indices].sort_index()

# Check that no row appears in both sets.
overlap = set(train_indices).intersection(set(test_indices))
assert len(overlap) == 0, "Train and test sets overlap."

# Print a short summary for learners.
print("Total rows:", row_count)
print("Training rows:", len(train_set))
print("Test rows:", len(test_set))
print("Test bridge IDs:", test_set["bridge_id"].to_list())
print("Train condition range:", train_set["condition"].min(), "to", train_set["condition"].max())
print("Test condition range:", test_set["condition"].min(), "to", test_set["condition"].max())



### **2.3. Train Test Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_02_03.jpg?v=1783555166" width="250">



>* Split data into training and testing sets
>* Test on unseen data to avoid overconfidence

>* Choose a suitable test data proportion
>* Randomly split while preserving enough training data

>* Keep features and targets correctly matched
>* Prevent leakage for trustworthy model evaluation



In [ ]:
#@title Python Code - Train Test Split

# Manual splitting protects honest model evaluation.
# Civil datasets need aligned feature targets.
# NumPy indices make splitting transparent.

# Import NumPy for small array operations.
import numpy as np

# Create reproducible concrete mix feature data.
rng = np.random.default_rng(seed=42)

# Store cement, water, age, and slump.
features = np.column_stack((
    rng.normal(350, 25, 12),
    rng.normal(180, 12, 12),
    rng.integers(7, 57, 12),
    rng.normal(8, 1.5, 12),
))

# Create matching compressive strength targets.
target = 0.08 * features[:, 0]
target += 0.45 * features[:, 2]
target -= 0.05 * features[:, 1]
target += rng.normal(0, 2, 12)

# Validate that rows remain aligned.
assert features.shape[0] == target.shape[0]

# Choose a simple twenty percent test split.
n_samples = features.shape[0]
test_fraction = 0.25
test_size = int(round(n_samples * test_fraction))

# Shuffle row indices reproducibly.
indices = np.arange(n_samples)
shuffled_indices = rng.permutation(indices)

# Slice shuffled indices into test and train.
test_indices = shuffled_indices[:test_size]
train_indices = shuffled_indices[test_size:]

# Apply identical indices to features and target.
X_train = features[train_indices]
y_train = target[train_indices]
X_test = features[test_indices]
y_test = target[test_indices]

# Check split sizes and alignment.
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert X_train.shape[0] + X_test.shape[0] == n_samples

# Print compact teaching results.
print("Manual train/test split using NumPy")
print(f"Total rows: {n_samples}")
print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")
print(f"Train indices: {train_indices.tolist()}")
print(f"Test indices: {test_indices.tolist()}")
print(f"First test strength: {y_test[0]:.1f} MPa")



## **3. Baseline Models**

### **3.1. Regression Baselines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_03_01.jpg?v=1783555153" width="250">



>* Predict the training average for numerical targets
>* Use baselines to check model learning

>* Baselines justify complex regression models
>* Mean and median reveal real improvement

>* Check models against simple predictions first
>* Use baselines to judge practical value



In [ ]:
#@title Python Code - Regression Baselines

# Regression baselines compare simple constant predictions.
# Civil engineering targets are often noisy.
# Mean and median baselines are useful.

# No extra installations are required.

# Import NumPy for small numerical arrays.
import numpy as np

# Make the example deterministic and reproducible.
rng = np.random.default_rng(42)

# Create synthetic bridge repair project features.
project_age_years = np.array([5, 8, 12, 15, 18, 22, 25, 30, 35, 40])

# Create repair costs in thousands of dollars.
repair_cost_k = np.array([42, 45, 50, 55, 60, 68, 72, 80, 95, 210])

# Validate matching feature and target lengths.
assert project_age_years.shape[0] == repair_cost_k.shape[0]

# Shuffle indices for a manual train test split.
indices = rng.permutation(repair_cost_k.size)

# Use seventy percent of projects for training.
train_size = int(0.7 * repair_cost_k.size)

# Separate shuffled indices into train and test.
train_idx = indices[:train_size]
test_idx = indices[train_size:]

# Select training and testing target values.
y_train = repair_cost_k[train_idx]
y_test = repair_cost_k[test_idx]

# Validate that both splits contain examples.
assert y_train.size > 0 and y_test.size > 0

# Build constant regression baseline predictions.
mean_prediction = np.mean(y_train)
median_prediction = np.median(y_train)

# Repeat each constant prediction for test cases.
mean_baseline = np.full(y_test.shape, mean_prediction)
median_baseline = np.full(y_test.shape, median_prediction)

# Define mean absolute error for regression evaluation.
def mean_absolute_error(actual_values, predicted_values):
    return np.mean(np.abs(actual_values - predicted_values))

# Evaluate both simple baseline predictors.
mean_mae = mean_absolute_error(y_test, mean_baseline)
median_mae = mean_absolute_error(y_test, median_baseline)

# Print compact results for classroom discussion.
print(f"Training projects: {y_train.size}, testing projects: {y_test.size}")
print(f"Mean baseline prediction: ${mean_prediction:.1f}k")
print(f"Median baseline prediction: ${median_prediction:.1f}k")
print(f"Mean baseline MAE: ${mean_mae:.1f}k")
print(f"Median baseline MAE: ${median_mae:.1f}k")
print("Lower MAE means the baseline is harder to beat.")



### **3.2. Classification Baselines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_03_02.jpg?v=1783555157" width="250">



>* Use simple baselines before complex classifiers
>* Compare against majority-class predictions on test data

>* Imbalanced classes can make accuracy misleading
>* Check errors to confirm meaningful learning

>* Use random, stratified, or rule-based baselines
>* Compare models against simple assumptions first



In [ ]:
#@title Python Code - Classification Baselines

# Classification baselines compare simple prediction rules.
# Civil datasets often have imbalanced classes.
# We evaluate baselines before complex classifiers.

import numpy as np
import pandas as pd

# Set a deterministic random seed.
rng = np.random.default_rng(42)

# Create small bridge inspection labels.
labels = np.array(
    ["low"] * 18 + ["moderate"] * 7 + ["high"] * 5
)

# Shuffle labels before splitting data.
indices = rng.permutation(len(labels))
labels = labels[indices]

# Make a manual train test split.
split_index = int(0.7 * len(labels))
y_train = labels[:split_index]
y_test = labels[split_index:]

# Validate the split sizes.
assert len(y_train) > 0 and len(y_test) > 0

# Find the majority training class.
classes, counts = np.unique(y_train, return_counts=True)
majority_class = classes[np.argmax(counts)]

# Predict the majority class always.
majority_pred = np.full(y_test.shape, majority_class)
majority_accuracy = np.mean(majority_pred == y_test)

# Predict randomly using training proportions.
probabilities = counts / counts.sum()
random_pred = rng.choice(classes, size=len(y_test), p=probabilities)
random_accuracy = np.mean(random_pred == y_test)

# Summarize class counts compactly.
train_counts = pd.Series(y_train).value_counts().to_dict()
test_counts = pd.Series(y_test).value_counts().to_dict()

# Print concise baseline results.
print("Training class counts:", train_counts)
print("Test class counts:", test_counts)
print("Majority baseline class:", majority_class)
print("Majority baseline accuracy:", round(majority_accuracy, 3))
print("Random frequency baseline accuracy:", round(random_accuracy, 3))
print("Compare future classifiers against these simple baselines.")



### **3.3. Regression Baselines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_03/Lecture_A/image_03_03.jpg?v=1783555155" width="250">



>* Predict the training average for numerical targets
>* Complex models should beat this minimum benchmark

>* Compare model gains against simple averages
>* Use medians when outliers distort costs

>* Train baselines fairly using training data only
>* Use baseline results to judge model value



In [ ]:
#@title Python Code - Regression Baselines

# Regression baselines compare against simple predictions.
# Civil engineering targets are often noisy.
# Use training data for fair baselines.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for repeatable splitting.
rng = np.random.default_rng(42)

# Create small concrete strength example data.
cement_kg = np.array([260, 280, 300, 320, 340, 360, 380, 400])
water_cm = np.array([18, 17, 16, 16, 15, 14, 14, 13])
strength_mpa = np.array([24, 27, 31, 33, 36, 39, 41, 45])

# Validate matching one dimensional arrays.
assert cement_kg.shape == strength_mpa.shape
assert water_cm.shape == strength_mpa.shape

# Shuffle row positions before splitting.
indices = rng.permutation(len(strength_mpa))
train_size = int(0.75 * len(indices))
train_idx = indices[:train_size]
test_idx = indices[train_size:]

# Separate target values for evaluation.
y_train = strength_mpa[train_idx]
y_test = strength_mpa[test_idx]

# Build mean and median regression baselines.
mean_value = np.mean(y_train)
median_value = np.median(y_train)
mean_pred = np.full(y_test.shape, mean_value)
median_pred = np.full(y_test.shape, median_value)

# Define simple error metrics manually.
mean_mae = np.mean(np.abs(y_test - mean_pred))
median_mae = np.mean(np.abs(y_test - median_pred))
mean_rmse = np.sqrt(np.mean((y_test - mean_pred) ** 2))
median_rmse = np.sqrt(np.mean((y_test - median_pred) ** 2))

# Print compact baseline results.
print("Regression baseline example: concrete strength")
print(f"Train cases: {len(y_train)}, test cases: {len(y_test)}")
print(f"Mean baseline prediction: {mean_value:.1f} MPa")
print(f"Median baseline prediction: {median_value:.1f} MPa")
print(f"Mean baseline MAE/RMSE: {mean_mae:.2f}/{mean_rmse:.2f} MPa")
print(f"Median baseline MAE/RMSE: {median_mae:.2f}/{median_rmse:.2f} MPa")

# Plot actual test values against baselines.
plt.figure(figsize=(6, 4))
plt.scatter(range(len(y_test)), y_test, label="Actual test strength")
plt.plot(range(len(y_test)), mean_pred, label="Mean baseline")
plt.plot(range(len(y_test)), median_pred, label="Median baseline")
plt.xlabel("Test case number")
plt.ylabel("Compressive strength, MPa")
plt.title("Regression baselines for concrete strength")
plt.legend()
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Workflow and Baselines**</font>


In this lecture, you learned to:
- Describe the steps in an end-to-end machine learning workflow for civil engineering data. 
- Implement a manual train/test split using simple Python and NumPy operations. 
- Build baseline predictors for regression and classification before training complex models. 

In the next Lecture (Lecture B), we will go over 'Metrics and Errors'